# 11 · Reprodução de Lira et al.: calibração (só o lado com vazamento)

Decide **como** reproduzir as Tabelas 4 e 5 antes de rodar as 59 arquiteturas. Tudo aqui usa a partição **com vazamento** (`vazado_p42`, a mesma do 04b); nenhuma rodada sem vazamento acontece neste notebook.

O repositório `leguwoi` tem mais de uma versão da receita, e o artigo descreve a mais antiga:

| Receita | Origem | Reamostragem | Pesos de classe | Scheduler | Épocas |
|---|---|---|---|---|---|
| **A** | tag `V_0.1` (19/02/2024), a que o texto do artigo descreve | SMOTEENN (semente `0x40`) | balanceados | cosseno com reinícios | 1000 |
| **B** | `master` (`83d82f1`) | nenhuma | uniformes | StepLR(50) | 300* |
| **B_smoteenn** | `master` com `--data_resampler SMOTEENN` | SMOTEENN (semente `0x40`) | uniformes | StepLR(50) | 300* |

\* O artigo diz 1000 épocas; com StepLR(50) o lr é 1e-8 na época 300. A etapa 3 confere que parar em 300 dá as mesmas predições.

**Regras fixadas antes de rodar:**
1. *Precisão:* AMP fp16 só é usado se, na ResNet-18, as predições de teste coincidirem em ≥ 98 % com as de fp32 e a acurácia diferir em ≤ 0,01.
2. *Épocas da receita B:* 300 se as predições na época 300 coincidirem em ≥ 99,5 % com as da época 1000 (AlexNet e ResNet-152); senão, 1000.
3. *Receita:* a de menor erro absoluto médio contra as linhas publicadas (acurácia, F1 macro e AUC macro de classes; ResNet-152, AlexNet e VGG-16; duas tarefas). O critério de sucesso é a ResNet-152 ficar a ≤ 0,03 da acurácia publicada (0,869 e 0,925); se nenhuma receita cumprir, isso fica registrado e a de menor erro é usada mesmo assim.

Semente: o leguwoi só usa `SEED = 0x40` na reamostragem. Partição, aumentações e pesos iniciais não têm semente no original; aqui recebem 0x40 (aumentações e pesos) e 42 (partição) para as rodadas serem repetíveis.

**Como rodar:** `TESTE_RAPIDO = True` na 1ª vez; depois `False` e **Executar tudo** (retoma sozinho se cair). Qualquer GPU serve: aqui não há métrica de tempo comparável. Use um ambiente com **RAM alta** (a reamostragem SMOTEENN trabalha com as imagens achatadas e passa de 12 GB).

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
RAIZ_TRABALHO = "/content"
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_MODULO_ESPERADO = "572e160bb80b3ce14863a8939eaee05f2fa21d85a6c13d6c71a7867e1a2b91fc"   # reproducao_lira.py
SHA_TABELAS_ESPERADO = "74cb1aedf91057c2cbf92896d02759a6d992bf7962e2ee18eb07f2b16d616d7c"  # dados/lira2024_tabelas_completas.csv
ABA = "A"
TESTE_RAPIDO = True          # 1ª vez True; depois False
MODELOS_CAL = ["Resnet152", "Alexnet", "VGG16"]
RECEITAS_CAL = ["B", "B_smoteenn", "A"]   # a A (1000 épocas) é a mais cara e fica por último

In [ ]:
# Versões do environment.yml do leguwoi onde o Colab permite; o que não instalar fica na versão do
# Colab e isso é registrado em cada resultado.json (campo "ambiente"). Rode ANTES de importar.
import subprocess, sys
def pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
pip("scikit-multilearn==0.2.0")
FIXOS = {
    "albumentations": ("1.4.0", "import numpy as np, albumentations as A; from albumentations.pytorch import ToTensorV2; "
        "A.Compose([A.RGBShift(p=1), A.UnsharpMask(p=1), A.SafeRotate(limit=30, p=1), A.GaussNoise(p=1), A.ISONoise(p=1), "
        "A.Sharpen(p=1), A.CLAHE(p=1), A.ElasticTransform(p=1), A.Resize(224, 224), A.Normalize(), ToTensorV2()])"
        "(image=np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8))"),
    "imbalanced-learn": ("0.12.0", "import numpy as np; from imblearn.combine import SMOTEENN; "
        "SMOTEENN(random_state=64).fit_resample(np.random.rand(80, 6), np.array([0] * 60 + [1] * 20))"),
    "skorch": ("0.15.0", "import numpy as np, torch, skorch; from skorch import NeuralNetClassifier; "
        "NeuralNetClassifier(torch.nn.Linear, module__in_features=4, module__out_features=2, max_epochs=1, verbose=0, "
        "criterion=torch.nn.CrossEntropyLoss, train_split=skorch.dataset.ValidSplit(cv=5, stratified=True))"
        ".fit(np.random.rand(40, 4).astype('float32'), np.array([0, 1] * 20))"),
}
for pacote, (versao, teste) in FIXOS.items():
    ok = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pacote}=={versao}"]).returncode == 0
    ok = ok and subprocess.run([sys.executable, "-c", teste], capture_output=True).returncode == 0
    if not ok:
        print(f"⚠️ {pacote}=={versao} não funciona neste Colab; instalando a versão atual (fica registrado).")
        pip("-U", pacote)
        r = subprocess.run([sys.executable, "-c", teste], capture_output=True, text=True)
        assert r.returncode == 0, f"{pacote} não funciona nem na versão atual:\n{r.stderr[-800:]}"
    else:
        print(f"{pacote}=={versao} OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, "O protocolo_comum.py do Drive não é o publicado."
ARQ_MODULO = f"{DIR_DRIVE}/reproducao_lira.py"
assert pc.sha256_arquivo(ARQ_MODULO) == SHA_MODULO_ESPERADO, (
    "O reproducao_lira.py do Drive não é o do repositório (SHA diferente). Envie de novo o arquivo certo.")
import reproducao_lira as rl
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar tipo → GPU (qualquer uma serve aqui)."
pc.definir_id_sessao(ABA)
print("GPU:", torch.cuda.get_device_name(0), "| módulo", rl.VERSAO, SHA_MODULO_ESPERADO[:16], "… | dono das travas:", pc.ID_SESSAO)

# Clona dataset e leguwoi nos commits fixados, importa o código do leguwoi e confere as sementes dele
ctx = rl.preparar(RAIZ_TRABALHO, caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}")
print("Sementes no leguwoi:", *ctx.meta["sementes_no_leguwoi"], sep="\n  ")
print("V_0.1 × master:", json.dumps(ctx.meta["diferencas_v01"], indent=1))
erros = rl.testar_metricas()
assert not erros, erros
print("Partição com vazamento = a do 04b:", rl.conferir_com_holdout_do_drive(ctx, f"{DIR_DRIVE}/dados/holdout_lira2024.csv"))
display(pd.DataFrame([rl.resumo_particao(ctx, t, p) for t in rl.TAREFAS for p in ("vazado_p42", "agrupado_d0")]))
print("Ambiente:", rl.ambiente())

R = f"{DIR_DRIVE}/resultados/reproducao_lira_calibracao"
C = f"{DIR_DRIVE}/checkpoints/reproducao_lira_calibracao"
RT = f"{DIR_DRIVE}/resultados_teste/reproducao_lira"
ARQ_TAB = f"{DIR_DRIVE}/dados/lira2024_tabelas_completas.csv"
assert pc.sha256_arquivo(ARQ_TAB) == SHA_TABELAS_ESPERADO, "lira2024_tabelas_completas.csv do Drive difere do repositório."
PUB = pd.read_csv(ARQ_TAB).set_index(["tarefa", "modelo_leguwoi"])

## 1 · Teste rápido (2 épocas, 160 imagens, pasta de teste)

In [ ]:
if TESTE_RAPIDO:
    shutil.rmtree(RT, ignore_errors=True)
    for receita, modelo, tarefa, prec in [("B", "Resnet18", "es", "fp32"), ("B", "Resnet18", "es", "amp"),
                                          ("A", "Alexnet", "dist", "fp32"), ("B_smoteenn", "MaxVit", "dist", "amp")]:
        r = rl.treinar_unidade(ctx, modelo, tarefa, "vazado_p42", receita, RT, None, precisao=prec, epocas=2,
                               limite_imagens=400, rotulo_extra=f"_{prec}")
        print(receita, modelo, tarefa, prec, "OK:", {k: round(r["metricas"][k], 3) for k in ("acc", "auc_M")},
              "| n_treino", r["dados"]["n_treino"], "| lote", r["lote"], f"| {r['treino_min']:.2f} min")
    print("➡️ Agora mude TESTE_RAPIDO = False e rode tudo de novo.")

## 2 · Ponte fp32 × AMP (ResNet-18, receita B, as duas tarefas)

In [ ]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar."
linhas = []
for tarefa in ("es", "dist"):
    par = {p: rl.treinar_unidade(ctx, "Resnet18", tarefa, "vazado_p42", "B", R, C, precisao=p, rotulo_extra=f"_ponte_{p}")
           for p in ("fp32", "amp")}
    conc = float(np.mean(np.array(par["fp32"]["y_pred"]) == np.array(par["amp"]["y_pred"])))
    linhas.append({"tarefa": tarefa, "acc_fp32": par["fp32"]["metricas"]["acc"], "acc_amp": par["amp"]["metricas"]["acc"],
                   "concordancia": conc, "min_fp32": par["fp32"]["treino_min"], "min_amp": par["amp"]["treino_min"]})
ponte = pd.DataFrame(linhas)
ponte["passa"] = (ponte["concordancia"] >= 0.98) & ((ponte["acc_fp32"] - ponte["acc_amp"]).abs() <= 0.01)
PRECISAO = "amp" if ponte["passa"].all() else "fp32"
display(ponte.round(4)); print("Precisão escolhida:", PRECISAO)

## 3 · Receita B: 300 × 1000 épocas (AlexNet e ResNet-152, entrada/saída)

In [ ]:
linhas = []
for modelo in ("Alexnet", "Resnet152"):
    r = rl.treinar_unidade(ctx, modelo, "es", "vazado_p42", "B", R, C, precisao=PRECISAO, epocas=1000,
                           instantaneos=(100, 200, 300, 400, 1000), rotulo_extra="_1000ep")
    fim = np.array(r["instantaneos"]["1000"])
    linhas.append({"modelo": modelo, **{f"conc_{e}": float(np.mean(np.array(r["instantaneos"][str(e)]) == fim))
                                        for e in (100, 200, 300, 400)}, "min_1000_epocas": r["treino_min"]})
ep = pd.DataFrame(linhas)
EPOCAS_B = 300 if (ep["conc_300"] >= 0.995).all() else 1000
display(ep.round(4)); print("Épocas da receita B:", EPOCAS_B)

## 4 · As três receitas nos três modelos (partição com vazamento)

In [ ]:
for receita in RECEITAS_CAL:
    fila = rl.montar_fila(receita, modelos=MODELOS_CAL, particoes=("vazado_p42",))
    kw = {"epocas": EPOCAS_B} if receita != "A" else {}
    rl.executar_fila(ctx, fila, R, C, precisao=PRECISAO, **kw)

## 4b · Diagnóstico: o que muda se os lotes forem embaralhados (fora da decisão)

O skorch não embaralha os lotes por padrão, e o CSV do dataset vem ordenado por classe (uma pasta depois da outra). No código público, portanto, quase todo lote de treino tem uma classe só. Esta etapa roda a receita B **com** embaralhamento (`B_embaralhada`, que **não** está no código público) nas ResNets pré-treinadas, só para medir quanto isso pesa. O resultado é mostrado e gravado, mas **não entra na escolha da receita**.

In [ ]:
MODELOS_DIAG = ["Resnet18", "Resnet152"]
rl.executar_fila(ctx, rl.montar_fila("B_embaralhada", modelos=MODELOS_DIAG, particoes=("vazado_p42",)), R, C,
                 precisao=PRECISAO, epocas=EPOCAS_B)
d = rl.ler_resultados(R)
d = d[d["receita"].isin(["B", "B_embaralhada"]) & d["modelo"].isin(MODELOS_DIAG) & (d["particao"] == "vazado_p42")]
diag = d[["receita", "tarefa", "modelo", "acc", "f1_M", "auc_M", "treino_min"]].copy()
for c in ("acc", "f1_M", "auc_M"):
    diag[f"pub_{c}"] = [PUB.loc[(t, m), c] for t, m in zip(diag["tarefa"], diag["modelo"])]
diag = diag.sort_values(["tarefa", "modelo", "receita"])
display(diag.round(3))

## 5 · Decisão (gravada antes de qualquer rodada sem vazamento)

In [ ]:
res = rl.ler_resultados(R)
res = res[res["receita"].isin(RECEITAS_CAL) & res["modelo"].isin(MODELOS_CAL) & (res["particao"] == "vazado_p42")]
faltam = len(RECEITAS_CAL) * len(MODELOS_CAL) * 2 - len(res)
assert faltam == 0, f"Faltam {faltam} unidades da calibração (veja erros acima ou rode de novo)."
COLS = ["acc", "f1_M", "auc_M"]
linhas = []
for _, r in res.iterrows():
    pub = PUB.loc[(r["tarefa"], r["modelo"])]
    linhas.append({"receita": r["receita"], "tarefa": r["tarefa"], "modelo": r["modelo"],
                   **{c: r[c] for c in COLS}, **{f"pub_{c}": pub[c] for c in COLS},
                   "erro_abs_medio": float(np.mean([abs(r[c] - pub[c]) for c in COLS])), "treino_min": r["treino_min"]})
cal = pd.DataFrame(linhas).sort_values(["tarefa", "modelo", "receita"])
display(cal.round(3))
placar = cal.groupby("receita")["erro_abs_medio"].mean().sort_values()
rn = cal[cal["modelo"] == "Resnet152"].assign(dif=lambda d: (d["acc"] - d["pub_acc"]).abs())
criterio = rn.groupby("receita")["dif"].max().le(0.03)
RECEITA = placar.index[0]
display(placar.round(4).to_frame("erro_abs_medio")); print("ResNet-152 a ≤ 0,03 da acurácia publicada:", criterio.to_dict())
print("Receita escolhida:", RECEITA, "| critério de sucesso cumprido:", bool(criterio[RECEITA]))

tempo = cal.groupby("receita")["treino_min"].sum()
decisao = {
    "gerado_em": pc.agora_iso(), "receita": RECEITA, "precisao": PRECISAO, "epocas_B": int(EPOCAS_B),
    "criterio_resnet152_cumprido": bool(criterio[RECEITA]), "placar_erro_abs_medio": placar.to_dict(),
    "ponte_amp": ponte.to_dict("records"), "epocas_300x1000": ep.to_dict("records"),
    "calibracao": cal.to_dict("records"), "diagnostico_embaralhamento": diag.to_dict("records"),
    "min_por_receita_3_modelos_2_tarefas": tempo.to_dict(),
    "regras": "ver célula de abertura do notebook 11", "modulo_sha256": SHA_MODULO_ESPERADO,
    "ambiente": rl.ambiente(), "contexto": ctx.meta,
}
arq = Path(DIR_DRIVE) / "resultados" / "calibracao_lira.json"
if arq.exists():
    antiga = json.loads(arq.read_text())
    assert (antiga["receita"], antiga["precisao"], antiga["epocas_B"]) == (RECEITA, PRECISAO, int(EPOCAS_B)), (
        "Já existe calibracao_lira.json com OUTRA decisão. Não sobrescrevo; confira antes de seguir.")
    print("calibracao_lira.json já existe com a mesma decisão.")
else:
    pc.gravar_json_atomico(decisao, arq); print("Gravado:", arq)

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*).